In [1]:
# Install or upgrade Google's official Gemini Python SDK
!pip install -q -U google-genai

print("✅ Gemini SDK installed successfully")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 984.2/984.2 kB 21.6 MB/s eta 0:00:00
✅ Gemini SDK installed successfully


In [2]:
from google import genai
from google.colab import userdata

# Securely retrieve the API key from Colab Secrets
try:
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

    if not GEMINI_API_KEY:
        raise ValueError("The API key is empty.")

    # Create the Gemini client
    client = genai.Client(api_key=GEMINI_API_KEY)

    print("✅ Gemini client configured successfully")

except Exception as error:
    print("❌ Could not load GEMINI_API_KEY.")
    print("Add it under Colab → Secrets and enable notebook access.")
    print("Error:", error)

✅ Gemini client configured successfully


In [3]:
# Model used throughout the demo
MODEL_NAME = "gemini-3.1-flash-lite"

test_prompt = """
You are an AI assistant for engineering students.

In exactly three short points, explain how an Agentic AI system
is different from a normal chatbot.
"""

try:
    response = client.interactions.create(
        model=MODEL_NAME,
        input=test_prompt
    )

    print("✅ Gemini API connection successful!\n")
    print(response.output_text)

except Exception as error:
    print("❌ Gemini API request failed.")
    print("Error:", error)

✅ Gemini API connection successful!

1. **Autonomy:** Unlike a chatbot that reacts to prompts, an agentic system breaks down high-level goals into multi-step plans and executes them independently.
2. **Tool Use:** While a chatbot is typically restricted to text generation, agentic AI can actively interface with external software, APIs, and databases to perform real-world actions.
3. **Iterative Reasoning:** Agentic systems use "self-reflection" loops to evaluate their progress, correct errors, and adjust their strategy dynamically until the task is complete.


In [4]:
# ---------------------------------------------------------
# SIMULATED ENGINEERING KNOWLEDGE BASE
# ---------------------------------------------------------

ENGINEERING_LIMITS = {
    "civil": {
        "concrete_bridge": {
            "maximum_crack_width_mm": 0.30,
            "maximum_deflection_mm": 20,
            "inspection_interval_days": 90
        }
    },

    "mechanical": {
        "industrial_motor": {
            "maximum_temperature_c": 80,
            "maximum_vibration_mm_s": 7.1,
            "minimum_lubrication_level_percent": 30
        }
    },

    "electrical": {
        "distribution_transformer": {
            "maximum_temperature_c": 95,
            "minimum_voltage_v": 210,
            "maximum_voltage_v": 250,
            "maximum_load_percent": 90
        }
    },

    "ece": {
        "communication_tower": {
            "minimum_signal_dbm": -100,
            "maximum_packet_loss_percent": 5,
            "maximum_latency_ms": 150
        }
    },

    "automobile": {
        "ev_battery": {
            "maximum_temperature_c": 50,
            "maximum_cell_imbalance_v": 0.05,
            "minimum_state_of_health_percent": 75
        }
    },

    "aerospace": {
        "aircraft_engine": {
            "maximum_vibration_mm_s": 6.0,
            "minimum_oil_pressure_psi": 40,
            "maximum_exhaust_temperature_c": 900
        }
    },

    "computer_science": {
        "application_server": {
            "maximum_cpu_percent": 85,
            "maximum_memory_percent": 90,
            "maximum_error_rate_percent": 3
        }
    }
}


# Display the available engineering branches
print("✅ Engineering knowledge base created\n")

print("Available branches:")
for number, branch in enumerate(ENGINEERING_LIMITS.keys(), start=1):
    print(f"{number}. {branch.replace('_', ' ').title()}")

✅ Engineering knowledge base created

Available branches:
1. Civil
2. Mechanical
3. Electrical
4. Ece
5. Automobile
6. Aerospace
7. Computer Science


In [5]:
# ---------------------------------------------------------
# TOOL 1: RETRIEVE ENGINEERING OPERATING LIMITS
# ---------------------------------------------------------

def get_operating_limits(domain: str, equipment: str) -> dict:
    """
    Retrieve simulated operating limits for a selected
    engineering domain and equipment type.

    Parameters
    ----------
    domain : str
        Engineering branch, such as mechanical or civil.

    equipment : str
        Equipment name, such as industrial_motor.

    Returns
    -------
    dict
        Equipment limits or an error message.
    """

    # Normalize user input
    normalized_domain = domain.strip().lower().replace(" ", "_")
    normalized_equipment = equipment.strip().lower().replace(" ", "_")

    # Check whether the domain exists
    if normalized_domain not in ENGINEERING_LIMITS:
        return {
            "status": "error",
            "message": f"Engineering domain '{domain}' was not found.",
            "available_domains": list(ENGINEERING_LIMITS.keys())
        }

    domain_equipment = ENGINEERING_LIMITS[normalized_domain]

    # Check whether the equipment exists
    if normalized_equipment not in domain_equipment:
        return {
            "status": "error",
            "message": (
                f"Equipment '{equipment}' was not found "
                f"under the '{domain}' domain."
            ),
            "available_equipment": list(domain_equipment.keys())
        }

    # Return the corresponding operating limits
    return {
        "status": "success",
        "domain": normalized_domain,
        "equipment": normalized_equipment,
        "limits": domain_equipment[normalized_equipment],
        "notice": (
            "These values are simulated for classroom demonstration "
            "and are not certified engineering standards."
        )
    }


# ---------------------------------------------------------
# TEST THE TOOL
# ---------------------------------------------------------

test_result = get_operating_limits(
    domain="Mechanical",
    equipment="Industrial Motor"
)

print("✅ Operating-limits tool created\n")
print(test_result)

✅ Operating-limits tool created

{'status': 'success', 'domain': 'mechanical', 'equipment': 'industrial_motor', 'limits': {'maximum_temperature_c': 80, 'maximum_vibration_mm_s': 7.1, 'minimum_lubrication_level_percent': 30}, 'notice': 'These values are simulated for classroom demonstration and are not certified engineering standards.'}


### Create the Engineering Risk Calculator Tool

In [6]:
# ---------------------------------------------------------
# TOOL 2: CALCULATE ENGINEERING RISK
# ---------------------------------------------------------

def calculate_risk_score(
    severity: int,
    likelihood: int,
    detectability: int
) -> dict:
    """
    Calculate a simplified engineering risk score.

    Each input must be between 1 and 10.

    severity:
        1 = negligible impact
        10 = catastrophic impact

    likelihood:
        1 = very unlikely
        10 = highly likely

    detectability:
        1 = easy to detect
        10 = difficult to detect
    """

    inputs = {
        "severity": severity,
        "likelihood": likelihood,
        "detectability": detectability
    }

    # Validate input types and ranges
    for name, value in inputs.items():
        if not isinstance(value, int):
            return {
                "status": "error",
                "message": f"{name} must be an integer."
            }

        if value < 1 or value > 10:
            return {
                "status": "error",
                "message": f"{name} must be between 1 and 10."
            }

    # Calculate Risk Priority Number
    risk_score = severity * likelihood * detectability

    # Assign a simplified risk category
    if risk_score >= 500:
        risk_level = "Critical"
        recommended_response = "Stop operation and escalate immediately."

    elif risk_score >= 250:
        risk_level = "High"
        recommended_response = "Take urgent corrective action."

    elif risk_score >= 100:
        risk_level = "Medium"
        recommended_response = "Investigate and schedule corrective action."

    else:
        risk_level = "Low"
        recommended_response = "Continue monitoring."

    return {
        "status": "success",
        "severity": severity,
        "likelihood": likelihood,
        "detectability": detectability,
        "risk_score": risk_score,
        "risk_level": risk_level,
        "recommended_response": recommended_response,
        "notice": (
            "This is a simplified classroom risk model and must not "
            "replace a certified engineering risk assessment."
        )
    }


# ---------------------------------------------------------
# TEST THE TOOL
# ---------------------------------------------------------

test_risk = calculate_risk_score(
    severity=9,
    likelihood=7,
    detectability=6
)

print("✅ Risk-calculation tool created\n")

for key, value in test_risk.items():
    print(f"{key}: {value}")

✅ Risk-calculation tool created

status: success
severity: 9
likelihood: 7
detectability: 6
risk_score: 378
risk_level: High
recommended_response: Take urgent corrective action.
notice: This is a simplified classroom risk model and must not replace a certified engineering risk assessment.


In [7]:
# ---------------------------------------------------------
# TOOL 3: RECORD AN APPROVED ENGINEERING ACTION
# ---------------------------------------------------------

from datetime import datetime, timezone

# Temporary in-memory action log
ACTION_LOG = []


def record_engineering_action(
    domain: str,
    equipment: str,
    recommended_action: str,
    priority: str,
    approved: bool
) -> dict:
    """
    Record an engineering action after human approval.

    Args:
        domain: Engineering branch, such as mechanical or civil.
        equipment: Equipment or infrastructure being assessed.
        recommended_action: Corrective or preventive action.
        priority: Low, Medium, High, or Critical.
        approved: True only when a responsible person approves the action.

    Returns:
        A dictionary describing whether the action was recorded.
    """

    valid_priorities = {"low", "medium", "high", "critical"}

    normalized_domain = domain.strip().lower().replace(" ", "_")
    normalized_equipment = equipment.strip().lower().replace(" ", "_")
    normalized_priority = priority.strip().lower()

    # Validate required text
    if not recommended_action.strip():
        return {
            "status": "error",
            "message": "The recommended action cannot be empty."
        }

    # Validate priority
    if normalized_priority not in valid_priorities:
        return {
            "status": "error",
            "message": (
                "Priority must be Low, Medium, High, or Critical."
            )
        }

    # Human approval gate
    if approved is not True:
        return {
            "status": "awaiting_human_approval",
            "message": (
                "The proposed action was not recorded because "
                "human approval is required."
            ),
            "proposed_action": recommended_action,
            "priority": normalized_priority.title()
        }

    # Create the approved record
    action_record = {
        "action_id": len(ACTION_LOG) + 1,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "domain": normalized_domain,
        "equipment": normalized_equipment,
        "recommended_action": recommended_action.strip(),
        "priority": normalized_priority.title(),
        "approved": True
    }

    ACTION_LOG.append(action_record)

    return {
        "status": "recorded",
        "message": "Engineering action recorded successfully.",
        "record": action_record
    }


# ---------------------------------------------------------
# TEST 1: ACTION WITHOUT APPROVAL
# ---------------------------------------------------------

pending_result = record_engineering_action(
    domain="Mechanical",
    equipment="Industrial Motor",
    recommended_action=(
        "Reduce the motor load and schedule an immediate inspection."
    ),
    priority="High",
    approved=False
)

print("TEST 1 — WITHOUT APPROVAL")
print(pending_result)


# ---------------------------------------------------------
# TEST 2: ACTION WITH APPROVAL
# ---------------------------------------------------------

approved_result = record_engineering_action(
    domain="Mechanical",
    equipment="Industrial Motor",
    recommended_action=(
        "Reduce the motor load and schedule an immediate inspection."
    ),
    priority="High",
    approved=True
)

print("\nTEST 2 — WITH APPROVAL")
print(approved_result)


# ---------------------------------------------------------
# DISPLAY THE ACTION LOG
# ---------------------------------------------------------

print("\nCURRENT ACTION LOG")
for action in ACTION_LOG:
    print(action)

TEST 1 — WITHOUT APPROVAL
{'status': 'awaiting_human_approval', 'message': 'The proposed action was not recorded because human approval is required.', 'proposed_action': 'Reduce the motor load and schedule an immediate inspection.', 'priority': 'High'}

TEST 2 — WITH APPROVAL
{'status': 'recorded', 'message': 'Engineering action recorded successfully.', 'record': {'action_id': 1, 'timestamp_utc': '2026-07-15T19:43:43.988827+00:00', 'domain': 'mechanical', 'equipment': 'industrial_motor', 'recommended_action': 'Reduce the motor load and schedule an immediate inspection.', 'priority': 'High', 'approved': True}}

CURRENT ACTION LOG
{'action_id': 1, 'timestamp_utc': '2026-07-15T19:43:43.988827+00:00', 'domain': 'mechanical', 'equipment': 'industrial_motor', 'recommended_action': 'Reduce the motor load and schedule an immediate inspection.', 'priority': 'High', 'approved': True}


### Connect Gemini to the Engineering Tools

In [8]:
# ---------------------------------------------------------
# CONNECT GEMINI TO THE ENGINEERING TOOLS
# ---------------------------------------------------------

from google.genai import types


ENGINEERING_AGENT_INSTRUCTION = """
You are an Agentic AI Engineering Incident Response Assistant.

Your responsibilities:

1. Understand the engineering incident.
2. Use get_operating_limits to retrieve the simulated equipment limits.
3. Compare the reported readings with those limits.
4. Select reasonable Severity, Likelihood, and Detectability ratings
   between 1 and 10.
5. Use calculate_risk_score to calculate the risk.
6. Explain why the ratings were selected.
7. Recommend immediate and preventive actions.
8. Clearly identify missing information.
9. Never claim that classroom limits are certified engineering standards.
10. Never execute or record an engineering action without human approval.

Return the final answer using these headings:

- Incident Summary
- Tools Used
- Threshold Comparison
- Risk Assessment
- Possible Causes
- Immediate Recommendations
- Preventive Recommendations
- Missing Information
- Human Approval Required
"""


# Sample mechanical-engineering incident
engineering_incident = """
Engineering domain: Mechanical
Equipment: Industrial Motor

Reported observations:
- Motor temperature: 98°C
- Vibration: 9.2 mm/s
- Lubrication level: 22%
- A slight grinding sound has been reported.
- Production management wants the motor to continue operating
  for another four hours.

Analyse the incident using the available engineering tools.
"""


try:
    engineering_response = client.models.generate_content(
        model=MODEL_NAME,
        contents=engineering_incident,
        config=types.GenerateContentConfig(
            system_instruction=ENGINEERING_AGENT_INSTRUCTION,

            # Gemini can decide when to call these Python functions
            tools=[
                get_operating_limits,
                calculate_risk_score
            ],

            # Allow several model-tool interaction steps
            automatic_function_calling=types.AutomaticFunctionCallingConfig(
                maximum_remote_calls=5
            ),

            temperature=0.2
        )
    )

    print("✅ ENGINEERING AGENT RESPONSE")
    print("=" * 80)
    print(engineering_response.text)

except Exception as error:
    print("❌ The engineering agent could not complete the analysis.")
    print("Error:", error)

✅ ENGINEERING AGENT RESPONSE
### Incident Summary
An industrial motor in the mechanical domain is exhibiting signs of significant distress. Reported conditions include an elevated operating temperature of 98°C, high vibration levels (9.2 mm/s), low lubrication levels (22%), and audible grinding noises. Production management has requested a four-hour extension of operation, which presents a high risk of catastrophic equipment failure.

### Tools Used
*   `get_operating_limits`: Used to retrieve baseline operational parameters for an industrial motor.
*   `calculate_risk_score`: Used to quantify the risk associated with continued operation under current conditions.

### Threshold Comparison
*Note: The following limits are based on simulated classroom data and are not certified engineering standards.*

| Parameter | Reported Value | Simulated Limit | Status |
| :--- | :--- | :--- | :--- |
| Temperature | 98°C | 85°C | **Exceeded** |
| Vibration | 9.2 mm/s | 4.5 mm/s | **Exceeded** |
| Lub

### Add the Human-Approval Gate

In [9]:
# ---------------------------------------------------------
# HUMAN-IN-THE-LOOP APPROVAL GATE
# ---------------------------------------------------------

# Remove the test entry created in Cell 7
ACTION_LOG.clear()

# Proposed action for the current motor incident
proposed_action = (
    "Safely reduce or stop the motor operation according to the site's "
    "approved procedure. Arrange inspection of lubrication, bearings, "
    "alignment, loading conditions, and vibration sources before restart."
)

proposed_priority = "High"

print("=" * 80)
print("PROPOSED ENGINEERING ACTION")
print("=" * 80)
print(f"Domain   : Mechanical")
print(f"Equipment: Industrial Motor")
print(f"Priority : {proposed_priority}")
print(f"Action   : {proposed_action}")

print("\n⚠️ Classroom simulation only.")
print("A qualified engineer must validate any real-world action.")

# Ask a human for approval
approval_input = input(
    "\nApprove recording this proposed action? "
    "Type YES to approve: "
).strip().lower()

human_approved = approval_input == "yes"

# Send the decision to the protected action-recording tool
approval_result = record_engineering_action(
    domain="Mechanical",
    equipment="Industrial Motor",
    recommended_action=proposed_action,
    priority=proposed_priority,
    approved=human_approved
)

print("\n" + "=" * 80)
print("APPROVAL RESULT")
print("=" * 80)

for key, value in approval_result.items():
    print(f"{key}: {value}")

print("\nCURRENT ACTION LOG")
print("-" * 80)

if ACTION_LOG:
    for action in ACTION_LOG:
        print(action)
else:
    print("No actions have been recorded.")

PROPOSED ENGINEERING ACTION
Domain   : Mechanical
Equipment: Industrial Motor
Priority : High
Action   : Safely reduce or stop the motor operation according to the site's approved procedure. Arrange inspection of lubrication, bearings, alignment, loading conditions, and vibration sources before restart.

⚠️ Classroom simulation only.
A qualified engineer must validate any real-world action.

Approve recording this proposed action? Type YES to approve: Yes

APPROVAL RESULT
status: recorded
message: Engineering action recorded successfully.
record: {'action_id': 1, 'timestamp_utc': '2026-07-15T19:47:40.874744+00:00', 'domain': 'mechanical', 'equipment': 'industrial_motor', 'recommended_action': "Safely reduce or stop the motor operation according to the site's approved procedure. Arrange inspection of lubrication, bearings, alignment, loading conditions, and vibration sources before restart.", 'priority': 'High', 'approved': True}

CURRENT ACTION LOG
-------------------------------------

### Add a Multi-Branch Engineering Scenario Selector

In [10]:
# ---------------------------------------------------------
# MULTI-BRANCH ENGINEERING INCIDENT SCENARIOS
# ---------------------------------------------------------

from google.genai import types


ENGINEERING_SCENARIOS = {
    1: {
        "branch": "Civil Engineering",
        "domain": "civil",
        "equipment": "concrete_bridge",
        "incident": """
Engineering domain: Civil
Infrastructure: Concrete Bridge

Reported observations:
- Current crack width: 0.80 mm
- Crack width during the previous inspection: 0.40 mm
- Measured deflection: 24 mm
- Heavy vehicles regularly use the bridge.
- Recent rainfall has caused water seepage near the crack.

Use the available tools to assess the incident, calculate risk,
identify possible causes, and recommend safe next steps.
"""
    },

    2: {
        "branch": "Mechanical Engineering",
        "domain": "mechanical",
        "equipment": "industrial_motor",
        "incident": """
Engineering domain: Mechanical
Equipment: Industrial Motor

Reported observations:
- Motor temperature: 98°C
- Vibration: 9.2 mm/s
- Lubrication level: 22%
- A grinding sound is present.
- Production wants the motor to run for four more hours.

Use the available tools to assess the incident, calculate risk,
identify possible causes, and recommend safe next steps.
"""
    },

    3: {
        "branch": "Electrical Engineering",
        "domain": "electrical",
        "equipment": "distribution_transformer",
        "incident": """
Engineering domain: Electrical
Equipment: Distribution Transformer

Reported observations:
- Transformer temperature: 108°C
- Output voltage fluctuates between 204 V and 256 V
- Current load: 97%
- A humming sound has become louder.
- The area is experiencing peak evening electricity demand.

Use the available tools to assess the incident, calculate risk,
identify possible causes, and recommend safe next steps.
"""
    },

    4: {
        "branch": "Electronics and Communication Engineering",
        "domain": "ece",
        "equipment": "communication_tower",
        "incident": """
Engineering domain: ECE
Equipment: Communication Tower

Reported observations:
- Signal strength: -112 dBm
- Packet loss: 12%
- Network latency: 240 ms
- Users report repeated call drops.
- Heavy rain and strong winds are present near the tower.

Use the available tools to assess the incident, calculate risk,
identify possible causes, and recommend safe next steps.
"""
    },

    5: {
        "branch": "Automobile Engineering",
        "domain": "automobile",
        "equipment": "ev_battery",
        "incident": """
Engineering domain: Automobile
Equipment: EV Battery

Reported observations:
- Battery temperature: 62°C
- Cell-voltage imbalance: 0.09 V
- State of health: 71%
- The driver reports unusually rapid battery discharge.
- The vehicle recently completed repeated fast-charging cycles.

Use the available tools to assess the incident, calculate risk,
identify possible causes, and recommend safe next steps.
"""
    },

    6: {
        "branch": "Aerospace Engineering",
        "domain": "aerospace",
        "equipment": "aircraft_engine",
        "incident": """
Engineering domain: Aerospace
Equipment: Aircraft Engine

Reported observations:
- Engine vibration: 8.4 mm/s
- Oil pressure: 34 psi
- Exhaust temperature: 925°C
- The abnormal readings appeared during a ground test.
- A metallic sound was reported near the engine.

Use the available tools to assess the incident, calculate risk,
identify possible causes, and recommend safe next steps.
"""
    },

    7: {
        "branch": "Computer Science Engineering",
        "domain": "computer_science",
        "equipment": "application_server",
        "incident": """
Engineering domain: Computer Science
System: Application Server

Reported observations:
- CPU utilisation: 96%
- Memory utilisation: 94%
- Application error rate: 8%
- User response time has increased significantly.
- A new software release was deployed one hour ago.

Use the available tools to assess the incident, calculate risk,
identify possible causes, and recommend safe next steps.
"""
    }
}


# ---------------------------------------------------------
# DISPLAY AVAILABLE SCENARIOS
# ---------------------------------------------------------

print("=" * 80)
print("AGENTIC AI ENGINEERING INCIDENT LAB")
print("=" * 80)

for scenario_number, scenario in ENGINEERING_SCENARIOS.items():
    print(f"{scenario_number}. {scenario['branch']}")


# ---------------------------------------------------------
# GET THE STUDENT'S SELECTION
# ---------------------------------------------------------

try:
    selected_number = int(
        input("\nSelect an engineering branch from 1 to 7: ")
    )

    if selected_number not in ENGINEERING_SCENARIOS:
        raise ValueError("Please select a number between 1 and 7.")

    selected_scenario = ENGINEERING_SCENARIOS[selected_number]

    print("\nSelected branch:", selected_scenario["branch"])
    print("Equipment:", selected_scenario["equipment"])

    print("\nRunning the engineering agent...")
    print("-" * 80)


    # ---------------------------------------------------------
    # RUN THE AGENT
    # ---------------------------------------------------------

    scenario_response = client.models.generate_content(
        model=MODEL_NAME,
        contents=selected_scenario["incident"],
        config=types.GenerateContentConfig(
            system_instruction=ENGINEERING_AGENT_INSTRUCTION,

            tools=[
                get_operating_limits,
                calculate_risk_score
            ],

            automatic_function_calling=(
                types.AutomaticFunctionCallingConfig(
                    maximum_remote_calls=6
                )
            ),

            temperature=0.2
        )
    )


    # ---------------------------------------------------------
    # DISPLAY THE FINAL RESPONSE
    # ---------------------------------------------------------

    print("\n" + "=" * 80)
    print(f"{selected_scenario['branch'].upper()} — AGENT ANALYSIS")
    print("=" * 80)

    print(scenario_response.text)


except ValueError as error:
    print("\n❌ Invalid selection:", error)

except Exception as error:
    print("\n❌ The engineering agent could not complete the scenario.")
    print("Error:", error)

AGENTIC AI ENGINEERING INCIDENT LAB
1. Civil Engineering
2. Mechanical Engineering
3. Electrical Engineering
4. Electronics and Communication Engineering
5. Automobile Engineering
6. Aerospace Engineering
7. Computer Science Engineering

Select an engineering branch from 1 to 7: 6

Selected branch: Aerospace Engineering
Equipment: aircraft_engine

Running the engineering agent...
--------------------------------------------------------------------------------

AEROSPACE ENGINEERING — AGENT ANALYSIS
### Incident Summary
During a ground test of an aircraft engine, the following abnormal parameters were observed: engine vibration of 8.4 mm/s, oil pressure of 34 psi, and an exhaust gas temperature (EGT) of 925°C. Additionally, a metallic sound was reported emanating from the engine area. These readings suggest a potential mechanical failure or internal component degradation.

### Tools Used
*   `get_operating_limits`: Used to retrieve simulated operational thresholds for an aircraft engine

### Add a Safety Reviewer Agent

In [11]:
# ---------------------------------------------------------
# AGENT 2: INDEPENDENT ENGINEERING SAFETY REVIEWER
# ---------------------------------------------------------

SAFETY_REVIEWER_INSTRUCTION = """
You are an independent Engineering Safety Reviewer.

Your job is to audit another AI agent's engineering incident analysis.

Review the analysis for:

1. Correct use of the reported sensor readings.
2. Correct comparison against simulated operating limits.
3. Reasonableness of the risk ratings.
4. Unsupported assumptions or hallucinations.
5. Missing safety considerations.
6. Whether immediate escalation is necessary.
7. Whether the recommendation clearly requires human approval.

Do not repeat the complete original response.

Return your review using exactly these headings:

- Review Decision
- What Was Done Well
- Problems or Missing Evidence
- Safety-Critical Concerns
- Required Corrections
- Final Recommendation

The Review Decision must be one of:

- ACCEPT
- ACCEPT WITH CAUTION
- REVISE
- ESCALATE IMMEDIATELY

Remember that all engineering limits are simulated classroom values.
Do not claim that the system is safe for real-world operation.
"""


try:
    # Confirm that Cell 10 was run successfully
    if "selected_scenario" not in globals():
        raise ValueError(
            "Run Cell 10 and select an engineering scenario first."
        )

    if "scenario_response" not in globals():
        raise ValueError(
            "No engineering-agent response was found. Run Cell 10 first."
        )

    original_agent_analysis = scenario_response.text

    reviewer_prompt = f"""
ENGINEERING INCIDENT
--------------------
{selected_scenario["incident"]}


ANALYSIS PRODUCED BY AGENT 1
----------------------------
{original_agent_analysis}


TASK
----
Independently audit Agent 1's analysis.

Check whether the diagnosis, threshold comparison, risk assessment,
recommended actions, and safety warnings are sufficiently supported
by the available information.
"""

    print("🔍 Safety Reviewer Agent is auditing the analysis...")
    print("-" * 80)

    safety_review_response = client.models.generate_content(
        model=MODEL_NAME,
        contents=reviewer_prompt,
        config=types.GenerateContentConfig(
            system_instruction=SAFETY_REVIEWER_INSTRUCTION,
            temperature=0.1
        )
    )

    # Save this for the next workflow step
    safety_review_text = safety_review_response.text

    print("\n" + "=" * 80)
    print("AGENT 2 — INDEPENDENT SAFETY REVIEW")
    print("=" * 80)
    print(safety_review_text)

except Exception as error:
    print("❌ The Safety Reviewer Agent could not complete the audit.")
    print("Error:", error)

🔍 Safety Reviewer Agent is auditing the analysis...
--------------------------------------------------------------------------------

AGENT 2 — INDEPENDENT SAFETY REVIEW
- Review Decision
REVISE

- What Was Done Well
The agent correctly identified that all three monitored parameters (vibration, oil pressure, and EGT) were outside of safe operating limits. The risk assessment methodology (Severity x Likelihood x Detectability) was applied logically, and the immediate recommendations (shutdown and securing the area) are appropriate for the reported symptoms.

- Problems or Missing Evidence
1. **Risk Calculation Error:** The agent calculated the risk score as 144 (9 * 8 * 2). Mathematically, 9 * 8 * 2 = 144. However, in standard Risk Priority Number (RPN) systems, "Detectability" is usually an inverse scale where a *lower* number indicates *higher* difficulty of detection. By assigning a 2 to "highly apparent" symptoms, the agent has penalized the risk score. If the symptoms are highly ap

In [12]:
# ---------------------------------------------------------
# AGENT 3: FINAL ENGINEERING DECISION COORDINATOR
# ---------------------------------------------------------

FINAL_COORDINATOR_INSTRUCTION = """
You are the Final Engineering Decision Coordinator.

You receive:

1. An engineering incident.
2. An analysis from a Diagnostic Agent.
3. An audit from an independent Safety Reviewer Agent.

Your role is to produce a concise, evidence-based final recommendation.

Rules:

- Give priority to safety-critical concerns raised by the reviewer.
- Do not invent measurements, standards, inspections, or test results.
- Clearly separate confirmed observations from possible causes.
- Treat all operating limits as simulated classroom values.
- Never declare real equipment or infrastructure safe to operate.
- Never execute or approve an engineering action.
- Clearly state that a qualified engineer must make the final decision.
- Recommend escalation when the available evidence indicates serious risk.

Return the answer using exactly these headings:

- Final Decision
- Confirmed Findings
- Final Risk Classification
- Immediate Recommended Action
- Additional Inspection Required
- Preventive Action
- Conditions Before Restart or Continued Operation
- Human Approval Status
"""


try:
    # ---------------------------------------------------------
    # VERIFY THAT THE PREVIOUS AGENTS HAVE RUN
    # ---------------------------------------------------------

    if "selected_scenario" not in globals():
        raise ValueError(
            "Run Cell 10 and select an engineering scenario first."
        )

    if "scenario_response" not in globals():
        raise ValueError(
            "Agent 1 analysis was not found. Run Cell 10 first."
        )

    if "safety_review_text" not in globals():
        raise ValueError(
            "Safety review was not found. Run Cell 11 first."
        )

    # ---------------------------------------------------------
    # PREPARE THE MULTI-AGENT CONTEXT
    # ---------------------------------------------------------

    coordinator_prompt = f"""
ENGINEERING BRANCH
------------------
{selected_scenario["branch"]}


ORIGINAL INCIDENT
-----------------
{selected_scenario["incident"]}


AGENT 1 — DIAGNOSIS AND RISK ANALYSIS
-------------------------------------
{scenario_response.text}


AGENT 2 — INDEPENDENT SAFETY REVIEW
-----------------------------------
{safety_review_text}


COORDINATION TASK
-----------------
Reconcile both agents' outputs.

Produce the safest final recommendation supported by the incident
data. Resolve any disagreement in favour of safety and clearly state
what requires human approval.
"""

    print("🤝 Final Decision Coordinator is combining both analyses...")
    print("-" * 80)

    # ---------------------------------------------------------
    # RUN AGENT 3
    # ---------------------------------------------------------

    final_coordination_response = client.models.generate_content(
        model=MODEL_NAME,
        contents=coordinator_prompt,
        config=types.GenerateContentConfig(
            system_instruction=FINAL_COORDINATOR_INSTRUCTION,
            temperature=0.1
        )
    )

    # Save values for the human-approval step
    final_recommendation_text = final_coordination_response.text

    final_domain = selected_scenario["domain"]
    final_equipment = selected_scenario["equipment"]

    # ---------------------------------------------------------
    # DISPLAY THE CONSOLIDATED RECOMMENDATION
    # ---------------------------------------------------------

    print("\n" + "=" * 80)
    print("AGENT 3 — FINAL COORDINATED RECOMMENDATION")
    print("=" * 80)

    print(final_recommendation_text)

    print("\n" + "=" * 80)
    print("CURRENT WORKFLOW STATUS")
    print("=" * 80)
    print("✅ Agent 1 completed diagnosis and risk analysis")
    print("✅ Agent 2 completed independent safety review")
    print("✅ Agent 3 prepared the consolidated recommendation")
    print("⏳ Waiting for human approval")
    print("\n⚠️ No action has been executed or recorded.")

except Exception as error:
    print("❌ The Final Decision Coordinator could not complete its task.")
    print("Error:", error)

🤝 Final Decision Coordinator is combining both analyses...
--------------------------------------------------------------------------------

AGENT 3 — FINAL COORDINATED RECOMMENDATION
### Final Decision
The engine must remain in a non-operational state. The combination of vibration, temperature, and pressure anomalies, coupled with audible metallic distress, indicates a high-probability mechanical failure. No further testing or diagnostic disassembly is permitted until a formal safety review and authorization by a qualified Lead Maintenance Engineer are completed.

### Confirmed Findings
*   **Vibration:** 8.4 mm/s (Exceeds simulated limit of 5.0 mm/s).
*   **Oil Pressure:** 34 psi (Below simulated limit of 40–60 psi).
*   **Exhaust Gas Temperature (EGT):** 925°C (Exceeds simulated limit of 850°C).
*   **Audible Evidence:** Metallic sound reported during ground test.
*   **Status:** All parameters indicate active mechanical degradation.

### Final Risk Classification
**Critical.** The 

In [13]:
# ---------------------------------------------------------
# FINAL HUMAN-IN-THE-LOOP APPROVAL
# ---------------------------------------------------------

try:
    # Verify that Cell 12 completed successfully
    if "final_recommendation_text" not in globals():
        raise ValueError(
            "Final recommendation not found. Run Cell 12 first."
        )

    if "final_domain" not in globals() or "final_equipment" not in globals():
        raise ValueError(
            "Scenario information not found. Run Cells 10–12 first."
        )

    print("=" * 80)
    print("FINAL ENGINEERING RECOMMENDATION FOR HUMAN REVIEW")
    print("=" * 80)

    print(f"\nDomain   : {final_domain.replace('_', ' ').title()}")
    print(f"Equipment: {final_equipment.replace('_', ' ').title()}")

    print("\n" + "-" * 80)
    print(final_recommendation_text)
    print("-" * 80)

    print(
        "\n⚠️ This is a classroom simulation. "
        "A qualified engineer must validate any real-world decision."
    )

    # ---------------------------------------------------------
    # HUMAN DECISION
    # ---------------------------------------------------------

    approval_input = input(
        "\nType APPROVE to record the recommendation, "
        "or REJECT to decline it: "
    ).strip().upper()

    if approval_input not in {"APPROVE", "REJECT"}:
        raise ValueError(
            "Invalid input. Enter either APPROVE or REJECT."
        )

    human_approved = approval_input == "APPROVE"

    # ---------------------------------------------------------
    # SELECT PRIORITY
    # ---------------------------------------------------------

    if human_approved:
        priority_input = input(
            "Enter priority "
            "(Low / Medium / High / Critical): "
        ).strip().title()

        valid_priorities = {"Low", "Medium", "High", "Critical"}

        if priority_input not in valid_priorities:
            raise ValueError(
                "Priority must be Low, Medium, High, or Critical."
            )

    else:
        priority_input = "High"

    # ---------------------------------------------------------
    # RECORD THROUGH THE PROTECTED TOOL
    # ---------------------------------------------------------

    final_approval_result = record_engineering_action(
        domain=final_domain,
        equipment=final_equipment,
        recommended_action=final_recommendation_text,
        priority=priority_input,
        approved=human_approved
    )

    print("\n" + "=" * 80)
    print("FINAL APPROVAL RESULT")
    print("=" * 80)

    for key, value in final_approval_result.items():
        print(f"{key}: {value}")

    # ---------------------------------------------------------
    # DISPLAY ACTION LOG
    # ---------------------------------------------------------

    print("\n" + "=" * 80)
    print("ENGINEERING ACTION LOG")
    print("=" * 80)

    if ACTION_LOG:
        for action in ACTION_LOG:
            print(f"\nAction ID : {action['action_id']}")
            print(f"Timestamp : {action['timestamp_utc']}")
            print(
                f"Domain    : "
                f"{action['domain'].replace('_', ' ').title()}"
            )
            print(
                f"Equipment : "
                f"{action['equipment'].replace('_', ' ').title()}"
            )
            print(f"Priority  : {action['priority']}")
            print(f"Approved  : {action['approved']}")
            print("Recommendation:")
            print(action["recommended_action"])
            print("-" * 80)

    else:
        print("No approved engineering actions have been recorded.")

except ValueError as error:
    print("\n❌ Input error:", error)

except Exception as error:
    print("\n❌ The approval workflow could not be completed.")
    print("Error:", error)

FINAL ENGINEERING RECOMMENDATION FOR HUMAN REVIEW

Domain   : Aerospace
Equipment: Aircraft Engine

--------------------------------------------------------------------------------
### Final Decision
The engine must remain in a non-operational state. The combination of vibration, temperature, and pressure anomalies, coupled with audible metallic distress, indicates a high-probability mechanical failure. No further testing or diagnostic disassembly is permitted until a formal safety review and authorization by a qualified Lead Maintenance Engineer are completed.

### Confirmed Findings
*   **Vibration:** 8.4 mm/s (Exceeds simulated limit of 5.0 mm/s).
*   **Oil Pressure:** 34 psi (Below simulated limit of 40–60 psi).
*   **Exhaust Gas Temperature (EGT):** 925°C (Exceeds simulated limit of 850°C).
*   **Audible Evidence:** Metallic sound reported during ground test.
*   **Status:** All parameters indicate active mechanical degradation.

### Final Risk Classification
**Critical.** The inc

In [14]:
# ---------------------------------------------------------
# INSTALL GRADIO FOR THE USER INTERFACE
# ---------------------------------------------------------

!pip install -q -U gradio

import gradio as gr

print("✅ Gradio installed successfully")
print("Gradio version:", gr.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.0/31.0 MB 76.4 MB/s eta 0:00:00
✅ Gradio installed successfully
Gradio version: 6.20.0


In [15]:
# ---------------------------------------------------------
# REUSABLE MULTI-AGENT ENGINEERING WORKFLOW
# ---------------------------------------------------------

# Create a lookup using the visible engineering branch name
BRANCH_LOOKUP = {
    scenario["branch"]: scenario
    for scenario in ENGINEERING_SCENARIOS.values()
}


def run_engineering_workflow(
    branch_name: str,
    custom_incident: str = ""
):
    """
    Run the complete three-agent engineering workflow.

    Args:
        branch_name:
            Engineering branch selected in the interface.

        custom_incident:
            Optional student-defined incident.
            When empty, the predefined branch scenario is used.

    Returns:
        Diagnostic analysis,
        safety review,
        final coordinated recommendation,
        workflow status.
    """

    try:
        # -----------------------------------------------------
        # VALIDATE THE SELECTED BRANCH
        # -----------------------------------------------------

        if not branch_name:
            raise ValueError("Select an engineering branch.")

        if branch_name not in BRANCH_LOOKUP:
            raise ValueError(
                f"No scenario is available for '{branch_name}'."
            )

        scenario = BRANCH_LOOKUP[branch_name]

        # Use the custom incident when supplied
        if custom_incident and custom_incident.strip():
            incident_details = custom_incident.strip()
        else:
            incident_details = scenario["incident"].strip()

        # Add reliable scenario metadata
        complete_incident = f"""
ENGINEERING BRANCH
------------------
{scenario["branch"]}

DOMAIN KEY
----------
{scenario["domain"]}

EQUIPMENT
---------
{scenario["equipment"]}

INCIDENT DETAILS
----------------
{incident_details}

Analyse this incident using the available engineering tools.
Use only the simulated classroom limits in the knowledge base.
"""

        # -----------------------------------------------------
        # AGENT 1 — DIAGNOSIS AND RISK ASSESSMENT
        # -----------------------------------------------------

        diagnosis_response = client.models.generate_content(
            model=MODEL_NAME,
            contents=complete_incident,
            config=types.GenerateContentConfig(
                system_instruction=ENGINEERING_AGENT_INSTRUCTION,
                tools=[
                    get_operating_limits,
                    calculate_risk_score
                ],
                temperature=0.2
            )
        )

        diagnosis_text = (
            diagnosis_response.text
            if diagnosis_response.text
            else "The Diagnostic Agent returned no text."
        )

        # -----------------------------------------------------
        # AGENT 2 — INDEPENDENT SAFETY REVIEW
        # -----------------------------------------------------

        reviewer_prompt = f"""
ORIGINAL ENGINEERING INCIDENT
-----------------------------
{complete_incident}


DIAGNOSTIC AGENT ANALYSIS
-------------------------
{diagnosis_text}


REVIEW TASK
-----------
Audit the analysis for:

- correct use of incident readings,
- correct comparison with simulated limits,
- reasonable risk ratings,
- unsupported assumptions,
- missing evidence,
- safety-critical concerns,
- need for qualified human approval.
"""

        review_response = client.models.generate_content(
            model=MODEL_NAME,
            contents=reviewer_prompt,
            config=types.GenerateContentConfig(
                system_instruction=SAFETY_REVIEWER_INSTRUCTION,
                temperature=0.1
            )
        )

        review_text = (
            review_response.text
            if review_response.text
            else "The Safety Reviewer returned no text."
        )

        # -----------------------------------------------------
        # AGENT 3 — FINAL DECISION COORDINATOR
        # -----------------------------------------------------

        coordinator_prompt = f"""
ENGINEERING INCIDENT
--------------------
{complete_incident}


AGENT 1 — DIAGNOSIS
-------------------
{diagnosis_text}


AGENT 2 — SAFETY REVIEW
-----------------------
{review_text}


COORDINATION TASK
-----------------
Produce the safest final recommendation supported by the evidence.

Resolve disagreements in favour of safety. Clearly identify the
actions that require approval from a qualified engineer.
"""

        coordinator_response = client.models.generate_content(
            model=MODEL_NAME,
            contents=coordinator_prompt,
            config=types.GenerateContentConfig(
                system_instruction=FINAL_COORDINATOR_INSTRUCTION,
                temperature=0.1
            )
        )

        final_text = (
            coordinator_response.text
            if coordinator_response.text
            else "The Decision Coordinator returned no text."
        )

        # -----------------------------------------------------
        # WORKFLOW STATUS
        # -----------------------------------------------------

        workflow_status = f"""
✅ Branch: {scenario["branch"]}

✅ Diagnostic Agent completed
✅ Engineering tools evaluated
✅ Safety Reviewer completed
✅ Decision Coordinator completed

⏳ Status: Waiting for human approval

⚠️ No physical engineering action has been executed.
"""

        return (
            diagnosis_text,
            review_text,
            final_text,
            workflow_status
        )

    except Exception as error:
        error_message = (
            "❌ The multi-agent workflow could not be completed.\n\n"
            f"Error: {error}"
        )

        return (
            error_message,
            error_message,
            error_message,
            "❌ Workflow failed"
        )


print("✅ Reusable multi-agent workflow created")
print("Available branches:")

for branch in BRANCH_LOOKUP:
    print("•", branch)

✅ Reusable multi-agent workflow created
Available branches:
• Civil Engineering
• Mechanical Engineering
• Electrical Engineering
• Electronics and Communication Engineering
• Automobile Engineering
• Aerospace Engineering
• Computer Science Engineering


In [16]:
# ---------------------------------------------------------
# BUILD THE GRADIO ENGINEERING AGENT APPLICATION
# ---------------------------------------------------------

import gradio as gr


def load_default_incident(branch_name):
    """Load the predefined incident for the selected branch."""

    if not branch_name or branch_name not in BRANCH_LOOKUP:
        return ""

    return BRANCH_LOOKUP[branch_name]["incident"].strip()


APP_CSS = """
.gradio-container {
    max-width: 1250px !important;
    margin: auto !important;
}

.header-box {
    text-align: center;
    padding: 20px;
    border-radius: 16px;
    margin-bottom: 16px;
}

.warning-box {
    padding: 12px;
    border-radius: 10px;
}
"""


with gr.Blocks(
    title="TensorLearners Engineering Agent",
    theme=gr.themes.Soft(),
    css=APP_CSS
) as engineering_app:

    # -----------------------------------------------------
    # APPLICATION HEADER
    # -----------------------------------------------------

    gr.Markdown(
        """
        # ⚙️ Agentic AI Engineering Incident Response System

        ### Built with Google Gemini and Python

        Select an engineering discipline, review the incident data,
        and run a multi-agent engineering assessment.

        **Workflow:** Diagnostic Agent → Safety Reviewer →
        Decision Coordinator → Human Approval
        """,
        elem_classes="header-box"
    )

    gr.Markdown(
        """
        > ⚠️ **Classroom demonstration only:**
        > The operating limits and recommendations in this application
        > are simulated. They must not replace professional engineering
        > standards, inspections, or decisions.
        """,
        elem_classes="warning-box"
    )

    # -----------------------------------------------------
    # INPUT SECTION
    # -----------------------------------------------------

    with gr.Row():

        with gr.Column(scale=1):

            branch_dropdown = gr.Dropdown(
                choices=list(BRANCH_LOOKUP.keys()),
                value=list(BRANCH_LOOKUP.keys())[0],
                label="Select Engineering Branch",
                info="Choose the engineering discipline to analyse."
            )

            load_button = gr.Button(
                "📋 Load Default Incident",
                variant="secondary"
            )

            run_button = gr.Button(
                "🚀 Run Multi-Agent Analysis",
                variant="primary"
            )

            clear_button = gr.ClearButton(
                value="🧹 Clear Results"
            )

        with gr.Column(scale=2):

            incident_input = gr.Textbox(
                value=load_default_incident(
                    list(BRANCH_LOOKUP.keys())[0]
                ),
                label="Engineering Incident",
                placeholder=(
                    "Enter equipment readings, observations, "
                    "environmental conditions, and operational concerns..."
                ),
                lines=15
            )

    # -----------------------------------------------------
    # WORKFLOW STATUS
    # -----------------------------------------------------

    workflow_status_output = gr.Markdown(
        value="### ⏳ Select a branch and run the analysis."
    )

    # -----------------------------------------------------
    # AGENT OUTPUTS
    # -----------------------------------------------------

    with gr.Tabs():

        with gr.Tab("🔍 Agent 1 — Diagnosis"):
            diagnosis_output = gr.Markdown(
                value="The diagnostic analysis will appear here."
            )

        with gr.Tab("🛡️ Agent 2 — Safety Review"):
            safety_output = gr.Markdown(
                value="The independent safety review will appear here."
            )

        with gr.Tab("🤝 Agent 3 — Final Decision"):
            final_output = gr.Markdown(
                value="The coordinated recommendation will appear here."
            )

    # -----------------------------------------------------
    # EVENT: LOAD DEFAULT INCIDENT
    # -----------------------------------------------------

    load_button.click(
        fn=load_default_incident,
        inputs=branch_dropdown,
        outputs=incident_input
    )

    branch_dropdown.change(
        fn=load_default_incident,
        inputs=branch_dropdown,
        outputs=incident_input
    )

    # -----------------------------------------------------
    # EVENT: RUN MULTI-AGENT WORKFLOW
    # -----------------------------------------------------

    run_button.click(
        fn=run_engineering_workflow,
        inputs=[
            branch_dropdown,
            incident_input
        ],
        outputs=[
            diagnosis_output,
            safety_output,
            final_output,
            workflow_status_output
        ],
        show_progress="full"
    )

    # Components cleared by the Clear button
    clear_button.add(
        components=[
            diagnosis_output,
            safety_output,
            final_output,
            workflow_status_output
        ]
    )


print("✅ Gradio application created")
print("🚀 Launching the application...")


# ---------------------------------------------------------
# LAUNCH FROM GOOGLE COLAB
# ---------------------------------------------------------

engineering_app.queue().launch(
    share=True,
    debug=True,
    show_error=True
)

/tmp/ipykernel_706/309160145.py:37: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


✅ Gradio application created
🚀 Launching the application...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2aedf0a1e3d22a6a22.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.


KeyboardInterrupt: 